# Baseline model: U-Net

## Notebook Setup

### pip installs

In [1]:
!pip install uv
!uv pip install --system -q -U numpy scipy torchmetrics wandb

!uv pip install --system git+https://github.com/JESvane99/DLVR_group15.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 46.0 MB/s eta 0:00:00
Using Python 3.13.16 environment at: /usr
Resolved 172 packages in 8.13s
⠙ Preparing packages... (0/17)                                                  warning: `build_system.requires = ["uv-build>=0.12.9,<0.13.0"]` does not contain the current uv version 0.13.3
Prepared 17 packages in 2.18s
Uninstalled 5 packages in 213ms
Installed 17 packages in 227ms
 + async-lru==2.4.0
 + dlvr==0.1.0 (from git+https://github.com/JESvane99/DLVR_group15.git@a065fccdfbcf11b23d3dec17315d1be9e0843c6f)
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.2.3
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.4
 + jupyterlab-server==2.28.1
 + loguru==0.7.3
 - matplotlib==3.10.0
 + matplotlib==3.11.2
 - pandas==2.2.3
 + pandas==3.0.6
 - pytest==8.4.2
 + pytest==9.1.1
 + ruff==0.17.1
 - scikit-learn==1.6.1
 + scikit-learn==1.9.1
 + segmentation-models-pytorch==0.5.1.dev0 (from git+https://github.com/qubvel/segmentation

### Imports

In [2]:
import copy
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchmetrics
import wandb
from google.colab import userdata
from segmentation_models_pytorch import Unet
from segmentation_models_pytorch.encoders import get_preprocessing_params
from torch.utils.data import DataLoader
from torchmetrics.classification import BinaryJaccardIndex
from torchvision import transforms

from dlvr.simple_crack_dataset import SimpleCrackDataset


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

### Helper methods


In [ ]:
def log_figure_to_wandb(fig, run: wandb.Run, save_name: str, caption: str | None = None):
    """
    Log a matplotlib figure to Weights & Biases.
    """
    run.log({save_name: wandb.Image(fig, caption=caption)})
    plt.close(fig)  # Close the figure to free memory

def mount_drive():
    cwd = Path.cwd()
    print(cwd.absolute())
    if not (cwd / 'drive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print("Drive already mounted.")
        print("drive contains:")
        print(*list((cwd / 'drive').glob('*')), sep='\n')
    return cwd / 'drive'

def dataset_path():
    """
    Return the path to the dataset directory.
    """
    dp = mount_drive() / 'MyDrive' / 'Colab_Shared_Datasets' / 'deepcrack_dataset'

    if not dp.exists():
        raise FileNotFoundError(f"Dataset path {dp} does not exist. Please check the path.")
    print(f"Dataset path: {dp}")
    print("Contents of the dataset directory:")
    print(*list(dp.glob('*')), sep='\n')
    return dp

def log_checkpoint_to_wandb(state_dict, run: wandb.Run, checkpoint_name: str = "best_baseline_model.pt"):
    """
    Log the model's state_dict as a Weights & Biases artifact.
    """
    artifact = wandb.Artifact(checkpoint_name, type="model")
    torch.save(state_dict, checkpoint_name)
    artifact.add_file(checkpoint_name)
    run.log_artifact(artifact, name=checkpoint_name, type="model", aliases=["baseline"])

def set_seed(value=42):
    """
    set the seed for everything that needs it
    """
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed(value)
    print(f"Seeds set to {value} for reproducibility.")

## Model Creation methods

### Optimizer choice

In [ ]:
def get_optimizer(model, learning_rate: float = 1e-3):
    """
    Return an Adam optimizer for the given model.
    """
    return torch.optim.Adam(model.parameters(), lr=learning_rate)

### Model choice

Check out Unet class in `segmentation_models_pytorch` for clarity on which default behaviors the model has.
important ones to mention as of now are:
* batchnormalization
* encoder depth of 5
* decoder interpolation being `nearest`
* decoder channels (see documentation)

In [ ]:
def get_model(device: torch.device):
    """
    Return a U-Net model with a ResNet34 encoder pretrained on ImageNet.
    """
    model = Unet(
        encoder_name="resnet34",        # Use a stable ResNet34 backbone
        encoder_weights="imagenet",     # Download weights pretrained on ImageNet
        in_channels=3,                  # RGB input channels
        classes=1,                # Number of output classes (1 for binary segmentation)
    )
    return model.to(device)  # Move the model to the specified device

### Preprocessing pipe

In [ ]:
def get_transforms(run) -> tuple[transforms.Compose, transforms.Compose]:
    """Define transformations for images and masks.

    returns: data_transform, target_transform"""
    preprocess_params = get_preprocessing_params(
        encoder_name="resnet34", pretrained="imagenet"
    )

    data_transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=preprocess_params["mean"], std=preprocess_params["std"]),
        ]
    )

    target_transform = transforms.Compose(
        [
            transforms.ToTensor(),
        ]
    )
    run.summary["preprocess_mean"] = preprocess_params["mean"]
    run.summary["preprocess_std"] = preprocess_params["std"]
    return data_transform, target_transform



### Dataloaders

In [ ]:

def get_train_val_dataloaders(data_path, split_ratio, batch_size, num_workers, pin_memory, run) -> tuple[DataLoader, DataLoader]:
    """build train, validation, and test dataloaders."""
    data_transform, target_transform = get_transforms(run)

    full_train_dataset = SimpleCrackDataset(
        data_path,
        mode="train",
        transform=data_transform,
        target_transform=target_transform,
    )
    val_ratio = split_ratio
    train_ratio = 1 - val_ratio
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_train_dataset, [train_ratio, val_ratio]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader

### Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = -1, mode: str = "max"):
        assert mode in ("max", "min")
        self.patience = patience
        self.mode = mode # if max the new value should be greater than current best
        self.best = None  # best monitored value so far
        self.best_epoch = 0  # epoch (0-based) at which it occurred
        self.best_state = None  # deep copy of the model's state_dict at that epoch
        self.bad_epochs = 0  # consecutive epochs without improvement

    def step(self, value, model, epoch):
        """Call once per epoch. Returns True if training should stop.
        value: the value of the monitored metric (e.g., validation IoU)
        model: the model being trained (to save its state_dict)
        epoch: the current epoch number (0-based)
        """
        if self.best is None:
            improved = True
        elif self.mode == "max":
            improved = value > self.best
        else:
            improved = value < self.best

        if improved:
            self.bad_epochs = 0
            self.best = value
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())

        else:
            self.bad_epochs += 1
        # True if training runs out of patience
        if self.patience < 0:
            return False  # no early stopping
        return self.patience <= self.bad_epochs



### Loss Function

No method defined as the loss function is loaded directly.
```python
nn.BCEWithLogitsLoss()
```
> The `BCEWithLogitsLoss` is used because it combines the sigmoid with the binary cross entropy
> in one step making it more stable than first calling the sigmoid and then the cross entropy

### Accuracy metric: Intersection over Union (Binary Jaccard Index)

No method defined as the accuracy metric IoU will be loaded directly

```python
iou_metric = BinaryJaccardIndex()
```



### Training step

In [ ]:
def training_step(model: torch.nn.Module,
                  optimizer: torch.optim.Optimizer,
                  loss_function: torch.nn.Module,
                  validation_metric: torchmetrics.Metric,
                  dataloader: DataLoader,
                  device: torch.device):
    """
    Perform one epoch of training.
    """
    model.train()  # Set the model to training mode
    running_loss = 0.0
    for images, masks in dataloader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        validation_metric.update(outputs, masks)

    mean_loss = running_loss / len(dataloader)
    mean_validation_metric = validation_metric.compute().item()
    return mean_loss, mean_validation_metric

### Validation Step

In [ ]:
def validation_step(model: torch.nn.Module,
                    loss_function: torch.nn.Module,
                    validation_metric: torchmetrics.Metric,
                    dataloader: DataLoader,
                    device: torch.device):
    """
    Perform one epoch of validation.
    """
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            outputs = model(images)
            loss = loss_function(outputs, masks)
            running_loss += loss.item()
            validation_metric.update(outputs, masks)

    mean_loss = running_loss / len(dataloader)
    mean_validation_metric = validation_metric.compute().item()
    return mean_loss, mean_validation_metric

## Configurations

### WandB config

In [ ]:

WANDB_ENTITY = "DLVR-group-15"
WANDB_PROJECT = "U_net_Crack_Segmentation"
WANDB_NAME = "Baseline"

### Training config

In [ ]:
config = {
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "datapath": dataset_path(),  # Mount Google Drive and get the dataset path
    "seeds": (420,),
    "learning_rate": 1e-3,
    "batch_size": 32,
    "num_workers": 2,  # Number of workers for DataLoader
    "num_epochs": 500,
    "early_stop_mode": "max",  # Monitor the validation IoU, so we want to maximize it
    "patience": -1,  # do not stop early, but keep track of the best model for logging
    "split_ratio": 0.2,  # Ratio of validation data to total data
    "pin_memory": True,  # Whether to use pinned memory for DataLoader
    "iou_threshold": 0.5,  # Threshold for IoU metric
}


## WandB login

In [ ]:
# Retrieve API key from Colab Secrets
wandb_key = userdata.get('WANDB-API-KEY')
wandb.login(key=wandb_key)

# RUN!

In [ ]:
for seed in config["seeds"]:
    set_seed(seed)
    with wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=WANDB_NAME + f"seed-{seed}",
        group=f"Baseline",
        config=config,
        tags=["Baseline"],
        notes="Baseline U-Net with ResNet34 encoder, trained on the DeepCrack dataset.",
    ) as run:
        train_loader, val_loader = get_train_val_dataloaders(
            data_path=config["datapath"],
            split_ratio=config["split_ratio"],  # 20% for validation
            batch_size=config["batch_size"],
            num_workers=config["num_workers"],
            pin_memory=config["pin_memory"],
            run=run
        )
        # loss function
        bce_loss_function = torch.nn.BCEWithLogitsLoss().to(config["device"])
        # validation metric IoU
        iou_metric = BinaryJaccardIndex(threshold=config["iou_threshold"]).to(config["device"])
        # model
        model = get_model(config["device"])
        # optimizer
        optimizer = get_optimizer(model, learning_rate=config["learning_rate"])
        # early stopping
        early_stopping = EarlyStopping(patience=config["patience"], mode=config["early_stop_mode"])

        for epoch in range(config["num_epochs"]):
            # Reset the validation metric at the start of each epoch
            iou_metric.reset()
            train_loss, train_iou = training_step(
                model, optimizer, bce_loss_function, iou_metric, train_loader, config["device"]
            )
            val_loss, val_iou = validation_step(
                model, bce_loss_function, iou_metric, val_loader, config["device"]
            )

            # Log metrics to wandb
            run.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_iou": train_iou,
                "val_loss": val_loss,
                "val_iou": val_iou,
            },
            step=epoch + 1,
            )

            stop_run = early_stopping.step(val_iou, model, epoch)
            if early_stopping.best_epoch == epoch:
                log_checkpoint_to_wandb(
                    early_stopping.best_state, run,
                    checkpoint_name="best_baseline_model.pt"
                )
                run.summary["best_val_iou"] = early_stopping.best
                run.summary["best_epoch"] = early_stopping.best_epoch + 1
            # Check for early stopping
            if stop_run:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break

